In [1]:
!poetry install -q

In [ ]:
"""
환경 설정 및 의존성 주입
- 목적: 프로젝트 경로 인식, 환경 변수 로드, 그리고 로컬 테스트를 위한 Docker DNS 우회
"""
import os
import sys
import datetime
import numpy as np
import pandas as pd
from dotenv import load_dotenv

# OpenMP 다중 로드 허용 및 스레드 경쟁 방지 환경 변수
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

# 1. 프로젝트 경로 설정 및 환경 변수 명시적 로드
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

# Docker 네트워크 외부(Host OS)에서 실행되는 Jupyter를 위한 DNS 해석 우회 처리
local_s3_endpoint = os.environ.get("LOCAL_S3_ENDPOINT", "")
if "localstack" in local_s3_endpoint:
    os.environ["LOCAL_S3_ENDPOINT"] = local_s3_endpoint.replace("localstack", "localhost")

✅ 환경 설정 및 모듈 임포트 완료


In [3]:
# DataFrame 출력 생략 방지 옵션 설정
pd.set_option('display.max_columns', None)        # 숨김 없이 모든 컬럼 출력
pd.set_option('display.max_colwidth', None)       # 컬럼 안의 긴 텍스트(Dict/List) 전체 출력
pd.set_option('display.expand_frame_repr', False) # 가로 너비 초과 시 줄바꿈 방지
pd.set_option('display.max_rows', 50)             # 필요시 최대 출력 행 수 조정

In [4]:
# ==============================================================================
# Gold Layer Multiverse Data Ingestion & Integration Engine
# ==============================================================================
"""S3 골드 레이어 파생 데이터셋 고속 수집 및 시계열 통합 엔진.

PyArrow Dataset API를 활용하여 S3(LocalStack)의 Hive 파티셔닝 Parquet 데이터를
C++ Native 엔진 수준에서 멀티스레드로 고속 읽기 및 인메모리 병합을 수행합니다.
"""

import os
import time
import boto3
from typing import List, Dict, Any, Optional

import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.fs as pafs

# [설계 의도] LocalStack S3 엔드포인트 환경 변수 바인딩
os.environ["LOCAL_S3_ENDPOINT"] = "http://localhost:4566"

# 1. LocalStack S3 Boto3 클라이언트 및 PyArrow Native S3FileSystem 초기화
# [설계 의도] Boto3는 S3 메타데이터 스캔(버킷 목록 조회)에 활용하고, 
# 고성능 Parquet 바이너리 I/O에는 PyArrow C++ Native S3FileSystem을 바인딩하여 네트워크 병목을 박멸함.
s3_client = boto3.client(
    "s3",
    endpoint_url="http://localhost:4566",
    aws_access_key_id="test",
    aws_secret_access_key="test",
    region_name="us-east-1"
)

s3_filesystem = pafs.S3FileSystem(
    endpoint_override="localhost:4566",
    access_key="test",
    secret_key="test",
    scheme="http",
    region="us-east-1"
)

bucket_name: str = "data-pipeline-gold"
base_prefix: str = "gold/market_data/"


def load_task_bucket_dataframe(
    bucket_name_str: str,
    task_name_str: str,
    bucket_id_str: str,
    s3_fs_instance: Optional[pafs.S3FileSystem] = None
) -> pd.DataFrame:
    """지정된 태스크(gold_daily_asia / gold_daily_global)와 버킷 ID 경로 하위의 모든 Parquet 파일들을
    PyArrow Dataset API로 스캔하고, S3 Hive 파티션 경로에서 trade_date를 복원하여 단일 데이터프레임으로 변환합니다.

    Args:
        bucket_name_str (str): 골드 S3 버킷 명칭.
        task_name_str (str): 실행 태스크 명칭 ('gold_daily_asia' 또는 'gold_daily_global').
        bucket_id_str (str): 파생 실험 버킷 식별자.
        s3_fs_instance (Optional[pafs.S3FileSystem]): PyArrow S3FileSystem 인스턴스. 미전달 시 글로벌 인스턴스 활용.

    Returns:
        pd.DataFrame: trade_date 컬럼이 추가된 일별 와이드 데이터프레임.
    """
    # [설계 의도] PyArrow S3FileSystem 의존성 바인딩 및 S3 대상 경로 설정
    fs_target: pafs.S3FileSystem = s3_fs_instance if s3_fs_instance is not None else s3_filesystem
    target_s3_path: str = f"{bucket_name_str}/gold/market_data/{task_name_str}/{bucket_id_str}"

    try:
        # PyArrow C++ Dataset 엔진 기반 S3 Parquet 고속 스캔 및 Hive 파티션 자동 탐색
        # year=YYYY/month=MM/day=DD 경로 구조를 C++ 레벨에서 자동으로 인식하여 컬럼화함
        dataset: ds.Dataset = ds.dataset(
            target_s3_path,
            filesystem=fs_target,
            format="parquet",
            partitioning="hive"
        )

        table: pa.Table = dataset.to_table()
        if table.num_rows == 0:
            return pd.DataFrame()

        # PyArrow Table을 단일 Pandas DataFrame으로 일괄 변환 (Python 객체 생성 오버헤드 제거)
        daily_dataframe: pd.DataFrame = table.to_pandas()

        # Hive 파티션(year, month, day)에서 trade_date 컬럼 벡터화 변환 및 생성
        if all(col in daily_dataframe.columns for col in ["year", "month", "day"]):
            daily_dataframe["trade_date"] = pd.to_datetime(
                daily_dataframe[["year", "month", "day"]]
            )
            daily_dataframe.drop(columns=["year", "month", "day"], inplace=True)

        return daily_dataframe

    except Exception:
        # [방어적 프로그래밍] 해당 경로에 객체가 없거나 예외 발생 시 빈 데이터프레임 반환
        return pd.DataFrame()


# ==============================================================================
# 2. S3 내 18종 파생 버킷 목록 자동 스캔 (gold_daily_asia 기준)
# ==============================================================================
scan_prefix: str = "gold/market_data/gold_daily_asia/"
response_scan = s3_client.list_objects_v2(
    Bucket=bucket_name,
    Prefix=scan_prefix,
    Delimiter="/"
)

available_bucket_job_ids: List[str] = []
if "CommonPrefixes" in response_scan:
    for prefix_info in response_scan["CommonPrefixes"]:
        folder_name: str = prefix_info["Prefix"].replace(scan_prefix, "").strip("/")
        if folder_name.startswith("bucket_"):
            available_bucket_job_ids.append(folder_name)

print("==========================================================================")
print(f" S3 골드 레이어 18종 파생 데이터셋(버킷) 스캔 완료 (총 {len(available_bucket_job_ids)}개)")
print("==========================================================================")

# 18개 버킷 전체를 결합하여 저장할 인메모리 데이터셋 레지스트리
gold_dataset_repository: Dict[str, pd.DataFrame] = {}

start_processing_time: float = time.time()

# ==============================================================================
# 3. 18종 데이터셋 순회 : Asia & Global 결합, trade_date 정렬 및 레지스트리 적재
# ==============================================================================
for dataset_index, bucket_job_id in enumerate(available_bucket_job_ids, start=1):
    print(f"[18/{dataset_index:02d}] {bucket_job_id:<70} ... ", end="")

    # daily_asia 데이터 로드 및 trade_date 추가 (PyArrow Dataset API 고속 처리)
    asia_market_dataframe: pd.DataFrame = load_task_bucket_dataframe(
        bucket_name_str=bucket_name,
        task_name_str="gold_daily_asia",
        bucket_id_str=bucket_job_id,
        s3_fs_instance=s3_filesystem
    )

    # daily_global 데이터 로드 및 trade_date 추가 (PyArrow Dataset API 고속 처리)
    global_market_dataframe: pd.DataFrame = load_task_bucket_dataframe(
        bucket_name_str=bucket_name,
        task_name_str="gold_daily_global",
        bucket_id_str=bucket_job_id,
        s3_fs_instance=s3_filesystem
    )

    # daily_asia와 daily_global 테이블 결합 (Outer Join)
    if not asia_market_dataframe.empty and not global_market_dataframe.empty:
        # trade_date 기준으로 두 시량 테이블 병합 (중복 컬럼 자동 병합 및 확장)
        combined_market_dataframe: pd.DataFrame = pd.merge(
            asia_market_dataframe,
            global_market_dataframe,
            on="trade_date",
            how="outer",
            suffixes=("", "_dup")
        )
        # 중복 사출된 컬럼 깔끔하게 정리
        duplicate_columns = [
            col for col in combined_market_dataframe.columns if col.endswith("_dup")
        ]
        if duplicate_columns:
            combined_market_dataframe.drop(columns=duplicate_columns, inplace=True)

    elif not asia_market_dataframe.empty:
        combined_market_dataframe = asia_market_dataframe
    else:
        combined_market_dataframe = global_market_dataframe

    # trade_date 기준으로 시간순 정렬 및 DatetimeIndex 바인딩
    if "trade_date" in combined_market_dataframe.columns:
        combined_market_dataframe.sort_values(by="trade_date", ascending=True, inplace=True)
        combined_market_dataframe.set_index("trade_date", inplace=True)

    # 완성된 결합 데이터셋 레지스트리에 저장
    gold_dataset_repository[bucket_job_id] = combined_market_dataframe
    print(f"DONE (Shape: {combined_market_dataframe.shape})")

elapsed_total_time: float = time.time() - start_processing_time
print("==========================================================================")
print(f" 18종 통합 골드 데이터셋 결합 완료 (총 소요시간: {elapsed_total_time:.2f}초)")
print("==========================================================================")

# ==============================================================================
# 4. [기능 4 구현] 최종 완성된 데이터셋 대표 출력 (첫 번째 버킷 기준)
# ==============================================================================
if available_bucket_job_ids:
    sample_bucket_id: str = available_bucket_job_ids[0]
    sample_final_dataset: pd.DataFrame = gold_dataset_repository[sample_bucket_id]

    print(f"\n[최종 데이터셋 대표 확인] Target Bucket: '{sample_bucket_id}'")
    if not sample_final_dataset.empty:
        print(f" - 시계열 인덱스 범위 : {sample_final_dataset.index.min()} ~ {sample_final_dataset.index.max()}")
        print(f" - 전체 행 개수 (거래일수) : {sample_final_dataset.shape[0]} 행")
        print(f" - 전체 피처(자산) 컬럼 수 : {sample_final_dataset.shape[1]} 개")

 S3 골드 레이어 18종 파생 데이터셋(버킷) 스캔 완료 (총 18개)
[18/01] bucket_impute_locf_detect_iqr_refine_clipping                          ... DONE (Shape: (10, 1488))
[18/02] bucket_impute_locf_detect_iqr_refine_masking                           ... DONE (Shape: (10, 1488))
[18/03] bucket_impute_locf_detect_isolation_forest_refine_clipping             ... DONE (Shape: (10, 1488))
[18/04] bucket_impute_locf_detect_isolation_forest_refine_masking              ... DONE (Shape: (10, 1488))
[18/05] bucket_impute_locf_detect_zscore_refine_clipping                       ... DONE (Shape: (10, 1488))
[18/06] bucket_impute_locf_detect_zscore_refine_masking                        ... DONE (Shape: (10, 1488))
[18/07] bucket_impute_log_return_detect_iqr_refine_clipping                    ... DONE (Shape: (10, 1488))
[18/08] bucket_impute_log_return_detect_iqr_refine_masking                     ... DONE (Shape: (10, 1488))
[18/09] bucket_impute_log_return_detect_isolation_forest_refine_clipping       ... DONE (Shape: